# Train **Hey Marantz**

This is Gerry's Marantz wake-word trainer. It uses the proven 2026 `saintpete/openwakeword-colab-toolkit` pipeline, pinned to commit `2e7379fd0980e83332d637c2f2b7496de48b2b41`, and changes only the wake-word-specific configuration.

**Use:** choose a T4 GPU, then **Runtime → Run all**. Keep the tab open. The training run can take around two hours.

The deployed phrase is **hey marantz**. The existing **Hey Jarvis** model remains the rollback model until this one has been tested on the media-server.


In [ ]:
# Cell 1 — fetch the known-good 2026 trainer and create the Hey Marantz variant.
import json, pathlib, shutil, subprocess, sys

UPSTREAM = 'https://github.com/saintpete/openwakeword-colab-toolkit.git'
PIN = '2e7379fd0980e83332d637c2f2b7496de48b2b41'
ROOT = pathlib.Path('/content/openwakeword-colab-toolkit')
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git', 'clone', '--quiet', UPSTREAM, str(ROOT)], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '--quiet', PIN], check=True)
src = ROOT / 'hey_clem_colab.ipynb'
nb = json.loads(src.read_text())

# Patch only wake-word-specific strings/configuration.
for cell in nb['cells']:
    text = ''.join(cell.get('source', []))
    text = text.replace('hey_clem', 'hey_marantz')
    text = text.replace('hey Clem', 'hey Marantz')
    text = text.replace('Hey Clem', 'Hey Marantz')
    text = text.replace("Clem's custom wake word", "Marantz custom wake word")
    text = text.replace('Clem', 'Marantz')
    text = text.replace('clem', 'marantz')
    cell['source'] = text.splitlines(keepends=True)

# Replace the embedded YAML target block with our single intended wake phrase.
for cell in nb['cells']:
    text = ''.join(cell.get('source', []))
    if 'target_phrase:' in text and 'model_name:' in text:
        start = text.index('target_phrase:')
        end = text.index('custom_negative_phrases:', start)
        text = text[:start] + 'target_phrase:\n  - \"hey marantz\"\n\n' + text[end:]
        # Replace inherited Clem near-misses with Marantz-oriented negatives.
        neg_start = text.index('custom_negative_phrases:')
        n_start = text.index('n_samples:', neg_start)
        negatives = '''custom_negative_phrases:
  - \"hey morantz\"
  - \"hey marants\"
  - \"hey marines\"
  - \"hey parents\"
  - \"hey miranda\"
  - \"marantz\"

'''
        text = text[:neg_start] + negatives + text[n_start:]
        # One target phrase: keep roughly the original 25k examples-per-phrase density.
        text = text.replace('n_samples: 75000', 'n_samples: 25000')
        text = text.replace('n_samples_val: 7500', 'n_samples_val: 2500')
        cell['source'] = text.splitlines(keepends=True)
        break

dst = pathlib.Path('/content/hey_marantz_colab_runtime.ipynb')
dst.write_text(json.dumps(nb, indent=1))
print('Prepared:', dst)
print('Pinned upstream commit:', PIN)
print('Target phrase: hey marantz')


In [ ]:
# Cell 2 — execute the patched trainer end-to-end in this GPU runtime.
# nbconvert executes the upstream cells in order, preserving its tested pipeline.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nbconvert'], check=True)
cmd = [
    'jupyter', 'nbconvert',
    '--to', 'notebook',
    '--execute', '/content/hey_marantz_colab_runtime.ipynb',
    '--output', '/content/hey_marantz_colab_executed.ipynb',
    '--ExecutePreprocessor.timeout=-1'
]
print('Starting Hey Marantz training. This is the long-running step...')
subprocess.run(cmd, check=True, cwd='/content')
print('Training notebook completed.')


In [ ]:
# Cell 3 — show the generated model/output files and offer the output ZIP for download.
from pathlib import Path
from google.colab import files
candidates = sorted(Path('/content').rglob('hey_marantz*'))
for p in candidates:
    if p.is_file():
        print(f'{p}  ({p.stat().st_size / 1024 / 1024:.2f} MiB)')
zips = [p for p in candidates if p.is_file() and p.suffix == '.zip']
if zips:
    print('Downloading:', zips[-1])
    files.download(str(zips[-1]))
else:
    print('No output ZIP found automatically. Check the file list above for the ONNX output.')
